In [19]:
# ============================================================
# TASK 11 — ENSEMBLE LEARNING
# ============================================================

import os
import sys
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    VotingClassifier,
    StackingClassifier
)
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_score

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

PROJECT_ROOT = os.path.abspath("..")

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("=======================================================")
print("        TASK 11 — ENSEMBLE LEARNING")
print("=======================================================")
print("Random seed:", RANDOM_STATE)
print("Project root:", PROJECT_ROOT)

        TASK 11 — ENSEMBLE LEARNING
Random seed: 42
Project root: /home/akash/Projects/Altrodav


In [20]:
from src.data import load_data, split_data

df = load_data()

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nTarget distribution:")
print(df["target"].value_counts())

X_train, X_val, X_test, y_train, y_val, y_test = split_data(df)

print("\n========== DATA SPLIT ==========")
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Dataset shape: (150, 5)

Columns:
['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)', 'target']

Target distribution:
target
0    50
1    50
2    50
Name: count, dtype: int64

========== DATA SPLIT ==========
Train: (105, 4)
Validation: (22, 4)
Test: (23, 4)


In [21]:
from src.features import (
    build_preprocessor,
    fit_preprocessor,
    transform_data
)

numerical_features = X_train.columns.tolist()

preprocessor = build_preprocessor(
    numerical_features
)

preprocessor = fit_preprocessor(
    preprocessor,
    X_train
)

X_train_processed = transform_data(
    preprocessor,
    X_train
)

X_val_processed = transform_data(
    preprocessor,
    X_val
)

X_test_processed = transform_data(
    preprocessor,
    X_test
)

print("========== PREPROCESSING ==========")
print("Train:", X_train_processed.shape)
print("Validation:", X_val_processed.shape)
print("Test:", X_test_processed.shape)
print("Preprocessing fitted only on training data.")

========== PREPROCESSING ==========
Train: (105, 4)
Validation: (22, 4)
Test: (23, 4)
Preprocessing fitted only on training data.


In [22]:
# ============================================================
# DIVERSE BASE MODELS
# ============================================================

logistic_model = LogisticRegression(
    C=1.0,
    max_iter=500,
    random_state=RANDOM_STATE
)

random_forest_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=5,
    min_samples_leaf=2,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

gradient_boosting_model = GradientBoostingClassifier(
    n_estimators=150,
    learning_rate=0.05,
    max_depth=2,
    min_samples_leaf=2,
    random_state=RANDOM_STATE
)

svm_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        SVC(
            C=1.0,
            kernel="rbf",
            probability=True,
            random_state=RANDOM_STATE
        )
    )
])

base_models = {
    "Logistic Regression": logistic_model,
    "Random Forest": random_forest_model,
    "Gradient Boosting": gradient_boosting_model,
    "RBF SVM": svm_model
}

print("========== BASE MODELS ==========")

for name in base_models:
    print("-", name)

========== BASE MODELS ==========
- Logistic Regression
- Random Forest
- Gradient Boosting
- RBF SVM


In [23]:
# ============================================================
# BASE MODEL VALIDATION PERFORMANCE
# ============================================================

base_validation_results = []

trained_base_models = {}

for name, model in base_models.items():

    fitted_model = clone(model)

    start = time.perf_counter()

    fitted_model.fit(
        X_train_processed,
        y_train
    )

    train_time = time.perf_counter() - start

    predictions = fitted_model.predict(
        X_val_processed
    )

    accuracy = accuracy_score(
        y_val,
        predictions
    )

    trained_base_models[name] = fitted_model

    base_validation_results.append({
        "Model": name,
        "Validation Accuracy": accuracy,
        "Training Time (s)": train_time
    })

base_validation_df = pd.DataFrame(
    base_validation_results
).sort_values(
    "Validation Accuracy",
    ascending=False
).reset_index(drop=True)

print("========== BASE MODEL VALIDATION ==========")
display(base_validation_df)

========== BASE MODEL VALIDATION ==========


,Model,Validation Accuracy,Training Time (s)
0,Logistic Regression,0.863636,0.008885
1,Random Forest,0.863636,0.746235
2,Gradient Boosting,0.863636,0.143524
3,RBF SVM,0.863636,0.003738


In [24]:
# ============================================================
# CROSS-VALIDATION
# ============================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

cv_results = []

for name, model in base_models.items():

    scores = cross_val_score(
        clone(model),
        X_train_processed,
        y_train,
        cv=cv,
        scoring="accuracy",
        n_jobs=-1
    )

    cv_results.append({
        "Model": name,
        "CV Mean Accuracy": scores.mean(),
        "CV Std": scores.std()
    })

cv_df = pd.DataFrame(
    cv_results
).sort_values(
    "CV Mean Accuracy",
    ascending=False
).reset_index(drop=True)

print("========== CROSS-VALIDATION ==========")
display(cv_df)

========== CROSS-VALIDATION ==========


,Model,CV Mean Accuracy,CV Std
0,Logistic Regression,0.980952,0.023328
1,RBF SVM,0.971429,0.038095
2,Random Forest,0.961905,0.035635
3,Gradient Boosting,0.952381,0.042592


In [25]:
# ============================================================
# SOFT VOTING ENSEMBLE
# ============================================================

voting_ensemble = VotingClassifier(
    estimators=[
        ("logistic", clone(logistic_model)),
        ("random_forest", clone(random_forest_model)),
        ("gradient_boosting", clone(gradient_boosting_model)),
        ("svm", clone(svm_model))
    ],
    voting="soft"
)

voting_scores = cross_val_score(
    voting_ensemble,
    X_train_processed,
    y_train,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

print("========== SOFT VOTING CV ==========")
print("Mean CV Accuracy:", voting_scores.mean())
print("CV Std:", voting_scores.std())

========== SOFT VOTING CV ==========
Mean CV Accuracy: 0.9523809523809523
CV Std: 0.04259177099999599


In [26]:
# ============================================================
# STACKING ENSEMBLE
# ============================================================

stacking_ensemble = StackingClassifier(
    estimators=[
        ("logistic", clone(logistic_model)),
        ("random_forest", clone(random_forest_model)),
        ("gradient_boosting", clone(gradient_boosting_model)),
        ("svm", clone(svm_model))
    ],
    final_estimator=LogisticRegression(
        max_iter=500,
        random_state=RANDOM_STATE
    ),
    cv=5,
    stack_method="predict_proba",
    n_jobs=-1
)

stacking_scores = cross_val_score(
    stacking_ensemble,
    X_train_processed,
    y_train,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

print("========== STACKING CV ==========")
print("Mean CV Accuracy:", stacking_scores.mean())
print("CV Std:", stacking_scores.std())

========== STACKING CV ==========
Mean CV Accuracy: 0.9523809523809523
CV Std: 0.04259177099999599


In [27]:
# ============================================================
# ENSEMBLE CV COMPARISON
# ============================================================

ensemble_cv_results = pd.DataFrame([
    {
        "Model": "Soft Voting Ensemble",
        "CV Mean Accuracy": voting_scores.mean(),
        "CV Std": voting_scores.std()
    },
    {
        "Model": "Stacking Ensemble",
        "CV Mean Accuracy": stacking_scores.mean(),
        "CV Std": stacking_scores.std()
    }
])

combined_cv = pd.concat(
    [
        cv_df,
        ensemble_cv_results
    ],
    ignore_index=True
).sort_values(
    "CV Mean Accuracy",
    ascending=False
).reset_index(drop=True)

print("========== ALL CV RESULTS ==========")
display(combined_cv)

========== ALL CV RESULTS ==========


,Model,CV Mean Accuracy,CV Std
0,Logistic Regression,0.980952,0.023328
1,RBF SVM,0.971429,0.038095
2,Random Forest,0.961905,0.035635
3,Gradient Boosting,0.952381,0.042592
4,Soft Voting Ensemble,0.952381,0.042592
5,Stacking Ensemble,0.952381,0.042592


In [28]:
# ============================================================
# SELECT BEST ENSEMBLE USING CV
# ============================================================

best_cv_model_name = combined_cv.iloc[0]["Model"]

print("Best CV configuration:")
print(best_cv_model_name)

if best_cv_model_name == "Soft Voting Ensemble":
    selected_ensemble = clone(voting_ensemble)

elif best_cv_model_name == "Stacking Ensemble":
    selected_ensemble = clone(stacking_ensemble)

else:
    selected_ensemble = None

print("\nSelection is based ONLY on cross-validation.")
print("The test set has not been used for model selection.")

Best CV configuration:
Logistic Regression

Selection is based ONLY on cross-validation.
The test set has not been used for model selection.


In [29]:
# ============================================================
# FINAL ENSEMBLE TRAINING
# ============================================================

if selected_ensemble is not None:

    start = time.perf_counter()

    selected_ensemble.fit(
        X_train_processed,
        y_train
    )

    ensemble_training_time = (
        time.perf_counter() - start
    )

    print("Selected ensemble trained successfully.")
    print(
        f"Training time: "
        f"{ensemble_training_time:.4f} seconds"
    )

else:
    print(
        "The best CV model is an individual model, "
        "so no ensemble was selected."
    )

The best CV model is an individual model, so no ensemble was selected.


In [30]:
# ============================================================
# VALIDATION COMPARISON
# ============================================================

validation_rows = []

for name, model in trained_base_models.items():

    predictions = model.predict(
        X_val_processed
    )

    validation_rows.append({
        "Model": name,
        "Validation Accuracy": accuracy_score(
            y_val,
            predictions
        )
    })

if selected_ensemble is not None:

    ensemble_val_predictions = (
        selected_ensemble.predict(
            X_val_processed
        )
    )

    ensemble_val_accuracy = accuracy_score(
        y_val,
        ensemble_val_predictions
    )

    validation_rows.append({
        "Model": best_cv_model_name,
        "Validation Accuracy": ensemble_val_accuracy
    })

validation_comparison = pd.DataFrame(
    validation_rows
).sort_values(
    "Validation Accuracy",
    ascending=False
).reset_index(drop=True)

print("========== VALIDATION COMPARISON ==========")
display(validation_comparison)

========== VALIDATION COMPARISON ==========


,Model,Validation Accuracy
0,Logistic Regression,0.863636
1,Random Forest,0.863636
2,Gradient Boosting,0.863636
3,RBF SVM,0.863636


In [31]:
# ============================================================
# FINAL UNSEEN TEST EVALUATION
# ============================================================

test_rows = []

for name, model in trained_base_models.items():

    predictions = model.predict(
        X_test_processed
    )

    test_rows.append({
        "Model": name,
        "Test Accuracy": accuracy_score(
            y_test,
            predictions
        )
    })

if selected_ensemble is not None:

    ensemble_test_predictions = (
        selected_ensemble.predict(
            X_test_processed
        )
    )

    ensemble_test_accuracy = accuracy_score(
        y_test,
        ensemble_test_predictions
    )

    test_rows.append({
        "Model": best_cv_model_name,
        "Test Accuracy": ensemble_test_accuracy
    })

test_df = pd.DataFrame(
    test_rows
).sort_values(
    "Test Accuracy",
    ascending=False
).reset_index(drop=True)

print("========== FINAL TEST RESULTS ==========")
display(test_df)

========== FINAL TEST RESULTS ==========


,Model,Test Accuracy
0,RBF SVM,1.000000
1,Logistic Regression,0.956522
2,Random Forest,0.956522
3,Gradient Boosting,0.956522


In [32]:
# ============================================================
# ENSEMBLE LIFT
# ============================================================

single_model_test_df = test_df[
    ~test_df["Model"].isin([
        "Soft Voting Ensemble",
        "Stacking Ensemble"
    ])
]

best_single_row = single_model_test_df.iloc[0]

best_single_name = best_single_row["Model"]
best_single_accuracy = best_single_row["Test Accuracy"]

ensemble_test_row = test_df[
    test_df["Model"] == best_cv_model_name
]

if len(ensemble_test_row) > 0:

    ensemble_accuracy = (
        ensemble_test_row["Test Accuracy"].iloc[0]
    )

    ensemble_lift = (
        ensemble_accuracy -
        best_single_accuracy
    )

else:

    ensemble_accuracy = np.nan
    ensemble_lift = np.nan

print("========== FINAL LIFT ==========")

print(
    f"Best Single Model      : "
    f"{best_single_name}"
)

print(
    f"Best Single Accuracy   : "
    f"{best_single_accuracy:.4f}"
)

print(
    f"Selected Ensemble      : "
    f"{best_cv_model_name}"
)

print(
    f"Ensemble Accuracy      : "
    f"{ensemble_accuracy:.4f}"
)

print(
    f"Ensemble Lift          : "
    f"{ensemble_lift:+.4f}"
)

========== FINAL LIFT ==========
Best Single Model      : RBF SVM
Best Single Accuracy   : 1.0000
Selected Ensemble      : Logistic Regression
Ensemble Accuracy      : 0.9565
Ensemble Lift          : -0.0435


In [33]:
# ============================================================
# MODEL DIVERSITY ANALYSIS
# ============================================================

prediction_data = {}

for name, model in trained_base_models.items():

    prediction_data[name] = model.predict(
        X_test_processed
    )

prediction_df = pd.DataFrame(
    prediction_data
)

print("========== PREDICTION AGREEMENT ==========")
display(
    prediction_df.corr()
)

print("\n========== PAIRWISE DISAGREEMENT ==========")

model_names = list(prediction_data.keys())

diversity_rows = []

for i in range(len(model_names)):

    for j in range(i + 1, len(model_names)):

        model_a = model_names[i]
        model_b = model_names[j]

        disagreement = np.mean(
            prediction_data[model_a]
            != prediction_data[model_b]
        )

        diversity_rows.append({
            "Model A": model_a,
            "Model B": model_b,
            "Disagreement": disagreement
        })

diversity_df = pd.DataFrame(
    diversity_rows
)

display(diversity_df)

========== PREDICTION AGREEMENT ==========


,Logistic Regression,Random Forest,Gradient Boosting,RBF SVM
Logistic Regression,1.000000,1.000000,0.928571,0.967495
Random Forest,1.000000,1.000000,0.928571,0.967495
Gradient Boosting,0.928571,0.928571,1.000000,0.967495
RBF SVM,0.967495,0.967495,0.967495,1.000000



========== PAIRWISE DISAGREEMENT ==========


,Model A,Model B,Disagreement
0,Logistic Regression,Random Forest,0.000000
1,Logistic Regression,Gradient Boosting,0.086957
2,Logistic Regression,RBF SVM,0.043478
3,Random Forest,Gradient Boosting,0.086957
4,Random Forest,RBF SVM,0.043478
5,Gradient Boosting,RBF SVM,0.043478


In [34]:
# ============================================================
# INFERENCE LATENCY
# ============================================================

models_for_timing = dict(
    trained_base_models
)

if selected_ensemble is not None:
    models_for_timing[
        best_cv_model_name
    ] = selected_ensemble

timing_rows = []

for name, model in models_for_timing.items():

    start = time.perf_counter()

    for _ in range(100):

        model.predict(
            X_test_processed
        )

    elapsed = time.perf_counter() - start

    average_ms = (
        elapsed / 100
    ) * 1000

    timing_rows.append({
        "Model": name,
        "Average Prediction Time (ms)": average_ms
    })

timing_df = pd.DataFrame(
    timing_rows
)

print("========== INFERENCE LATENCY ==========")
display(timing_df)

========== INFERENCE LATENCY ==========


,Model,Average Prediction Time (ms)
0,Logistic Regression,0.122411
1,Random Forest,66.692199
2,Gradient Boosting,0.544853
3,RBF SVM,0.181138


In [35]:
# ============================================================
# TASK 11 — FINAL DECISION
# ============================================================

print("=======================================================")
print("        TASK 11 — ENSEMBLE LEARNING")
print("=======================================================")

print(
    f"\nBest Single Model      : "
    f"{best_single_name}"
)

print(
    f"Best Single Accuracy   : "
    f"{best_single_accuracy:.4f}"
)

print(
    f"Selected Ensemble      : "
    f"{best_cv_model_name}"
)

print(
    f"Ensemble Accuracy      : "
    f"{ensemble_accuracy:.4f}"
)

print(
    f"Ensemble Lift          : "
    f"{ensemble_lift:+.4f}"
)

if ensemble_lift > 0:

    print("\nFINAL DECISION:")
    print(
        "The ensemble beats the best single model "
        "on the unseen test set."
    )
    print(
        "The ensemble therefore provides measurable "
        "performance improvement."
    )

else:

    print("\nFINAL DECISION:")
    print(
        "The ensemble does not beat the best single "
        "model on the unseen test set."
    )
    print(
        "The implementation is valid, but the core "
        "performance requirement is not satisfied."
    )

        TASK 11 — ENSEMBLE LEARNING

Best Single Model      : RBF SVM
Best Single Accuracy   : 1.0000
Selected Ensemble      : Logistic Regression
Ensemble Accuracy      : 0.9565
Ensemble Lift          : -0.0435

FINAL DECISION:
The ensemble does not beat the best single model on the unseen test set.
The implementation is valid, but the core performance requirement is not satisfied.


In [36]:
# ============================================================
# SAVE TASK 11 ARTIFACTS
# ============================================================

LOG_DIR = os.path.join(
    PROJECT_ROOT,
    "logs"
)

os.makedirs(
    LOG_DIR,
    exist_ok=True
)

final_results = test_df.copy()

final_results["Best Single Model"] = best_single_name
final_results["Best Single Accuracy"] = best_single_accuracy
final_results["Selected Ensemble"] = best_cv_model_name
final_results["Ensemble Lift"] = ensemble_lift

results_path = os.path.join(
    LOG_DIR,
    "task11_ensemble_results.csv"
)

final_results.to_csv(
    results_path,
    index=False
)

cv_path = os.path.join(
    LOG_DIR,
    "task11_cv_results.csv"
)

combined_cv.to_csv(
    cv_path,
    index=False
)

diversity_path = os.path.join(
    LOG_DIR,
    "task11_diversity_results.csv"
)

diversity_df.to_csv(
    diversity_path,
    index=False
)

latency_path = os.path.join(
    LOG_DIR,
    "task11_latency_results.csv"
)

timing_df.to_csv(
    latency_path,
    index=False
)

print("========== ARTIFACTS SAVED ==========")

print(results_path)
print(cv_path)
print(diversity_path)
print(latency_path)

========== ARTIFACTS SAVED ==========
/home/akash/Projects/Altrodav/logs/task11_ensemble_results.csv
/home/akash/Projects/Altrodav/logs/task11_cv_results.csv
/home/akash/Projects/Altrodav/logs/task11_diversity_results.csv
/home/akash/Projects/Altrodav/logs/task11_latency_results.csv


In [37]:
# ============================================================
# FINAL TASK 11 VERIFICATION
# ============================================================

print("=======================================================")
print("        TASK 11 — FINAL VERIFICATION")
print("=======================================================")

print("\nDataset:")
print(" - Iris dataset")
print(" - Stratified train/validation/test split")
print(" - Fixed random seed = 42")

print("\nBase Models:")
for name in base_models:
    print(" -", name)

print("\nEnsemble Methods:")
print(" - Soft Voting")
print(" - Stacking")

print("\nEvaluation:")
print(" - Cross-validation")
print(" - Validation set")
print(" - Unseen test set")

print("\nAnalysis:")
print(" - Prediction diversity")
print(" - Inference latency")
print(" - Accuracy lift")

print("\nBest Single Model:")
print(best_single_name)

print(
    "Best Single Accuracy:",
    f"{best_single_accuracy:.4f}"
)

print("\nSelected Ensemble:")
print(best_cv_model_name)

print(
    "Ensemble Accuracy:",
    f"{ensemble_accuracy:.4f}"
)

print(
    "Ensemble Lift:",
    f"{ensemble_lift:+.4f}"
)

print("\nArtifacts:")
print("-", results_path)
print("-", cv_path)
print("-", diversity_path)
print("-", latency_path)

print("\n=======================================================")

if ensemble_lift > 0:

    print(
        "TASK 11 COMPLETED SUCCESSFULLY!"
    )

else:

    print(
        "TASK 11 IMPLEMENTATION COMPLETED, "
        "BUT THE ENSEMBLE DID NOT BEAT THE BEST "
        "SINGLE MODEL."
    )

print("=======================================================")

        TASK 11 — FINAL VERIFICATION

Dataset:
 - Iris dataset
 - Stratified train/validation/test split
 - Fixed random seed = 42

Base Models:
 - Logistic Regression
 - Random Forest
 - Gradient Boosting
 - RBF SVM

Ensemble Methods:
 - Soft Voting
 - Stacking

Evaluation:
 - Cross-validation
 - Validation set
 - Unseen test set

Analysis:
 - Prediction diversity
 - Inference latency
 - Accuracy lift

Best Single Model:
RBF SVM
Best Single Accuracy: 1.0000

Selected Ensemble:
Logistic Regression
Ensemble Accuracy: 0.9565
Ensemble Lift: -0.0435

Artifacts:
- /home/akash/Projects/Altrodav/logs/task11_ensemble_results.csv
- /home/akash/Projects/Altrodav/logs/task11_cv_results.csv
- /home/akash/Projects/Altrodav/logs/task11_diversity_results.csv
- /home/akash/Projects/Altrodav/logs/task11_latency_results.csv

TASK 11 IMPLEMENTATION COMPLETED, BUT THE ENSEMBLE DID NOT BEAT THE BEST SINGLE MODEL.
